# ASL Fingerspelling Recognizer

Trains a classifier on MediaPipe hand landmarks from the
[ASL Alphabet dataset](https://www.kaggle.com/datasets/grassknoted/asl-alphabet).

Outputs `asl_model.zip`, a TF.js model for A–Z fingerspelling classification.

## Setup

Install the only package we need that isn’t already on Kaggle: MediaPipe for hand landmark extraction.

In [ ]:
!pip install -qq --no-warn-conflict mediapipe

Download MediaPipe's Hand Landmarker model.

In [ ]:
import os, urllib.request
from tqdm.notebook import tqdm

url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
path = "hand_landmarker.task"

if not os.path.exists(path):
    with tqdm(unit='B', unit_scale=True, desc="Downloading") as bar:
        def report(blocks, size, total):
            bar.total = total
            bar.n = min(blocks * size, total) 
            bar.refresh()
            
        urllib.request.urlretrieve(url, path, reporthook=report)

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import mediapipe as mp
import cv2
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import shutil
from tqdm.auto import tqdm

## Download the dataset

KaggleHub downloads the ASL Alphabet dataset and caches it locally.

In [ ]:
import kagglehub
DATA_DIR = Path(kagglehub.dataset_download("grassknoted/asl-alphabet"))
print(f"Dataset ready at {DATA_DIR}")

## Explore the dataset

The ASL Alphabet dataset contains ~87,000 images of hands spelling A–Z.
Each image is 200×200 pixels with a subject’s hand forming one letter against a dark background.

In [ ]:
from glob import glob
import pandas as pd

train_path = str(DATA_DIR / "asl_alphabet_train" / "asl_alphabet_train")
labels = [parent.split("/")[-1] for parent in glob(train_path + "/*")]
labels.sort()

# df = pd.DataFrame(columns=["label", "image", "landmarks"])
# for label in labels:
#     images = [filename.split("/")[-1] for filename in glob(f"{train_path}/{label}/*")]
#     images.sort()
#     df = pd.concat(
#         [df, pd.DataFrame([{"image": image, "label": label} for image in images])],
#         ignore_index=True
#     )
# df.to_csv("./train.csv", index=False)

df = pd.read_csv("./train.csv")

In [ ]:
unique_samples = df.drop_duplicates(subset=['label']).sort_values('label')

fig, axes = plt.subplots(5, 6, figsize=(12, 12))
flat_axes = axes.flatten()

for ax, (idx, row) in zip(flat_axes, unique_samples.iterrows()):
    img = plt.imread(f"{train_path}/{row["label"]}/{row["image"]}")
    ax.imshow(img)
    ax.set_title(str(row["label"]))

for ax in flat_axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

## Extract hand landmarks

MediaPipe draws 21 landmarks on each detected hand (fingertips, knuckles, wrist).
We normalise by subtracting the wrist position so the features are translation-invariant.

In [ ]:
# Utilities

import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=1)
detector = vision.HandLandmarker.create_from_options(options)


def detect_landmarks(img):
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img)
    result = detector.detect(mp_image)
    return result


import dataclasses
import cv2
import numpy as np
import mediapipe as mp

mp_hands = mp.tasks.vision.HandLandmarksConnections
mp_drawing = mp.tasks.vision.drawing_utils
mp_drawing_styles = mp.tasks.vision.drawing_styles


def  draw_landmarks_on_image(rgb_image, detection_result):
    annotated_image = np.copy(rgb_image)
    hand_landmarks_style = {
        k: dataclasses.replace(v, circle_radius=2)
        for k, v in mp_drawing_styles.get_default_hand_landmarks_style().items()
    }
    hand_connections_style = {
        k: dataclasses.replace(v, thickness=1)
        for k, v in mp_drawing_styles.get_default_hand_connections_style().items()
    }
    for idx in range(len(detection_result.hand_landmarks)):
        hand_landmarks = detection_result.hand_landmarks[idx]
        handedness = detection_result.handedness[idx]
        mp_drawing.draw_landmarks(
            annotated_image,
            hand_landmarks,
            mp_hands.HAND_CONNECTIONS,
            hand_landmarks_style,
            hand_connections_style,
        )
    return annotated_image

In [ ]:
# # Compute and cache results

# import pickle
# import cv2

# from tqdm.notebook import tqdm
# from IPython.display import display
# from PIL import Image

# preview = display(display_id=True)
# pkl_root = "train/pkl"

# def pkl_path_for(row):
#     return f"{pkl_root}/{row.label}/{row.image}.pkl"

# def is_cached(pkl_path):
#     if not os.path.exists(pkl_path):
#         return False
#     try:
#         with open(pkl_path, "rb") as f:
#             pickle.load(f)
#         return True
#     except Exception:
#         return False

# def atomic_write_pkl(pkl_path, obj):
#     os.makedirs(os.path.dirname(pkl_path), exist_ok=True)
#     tmp = pkl_path + ".tmp"
#     with open(tmp, "wb") as f:
#         pickle.dump(obj, f)
#     os.replace(tmp, pkl_path)

# misses = []
# for row in tqdm(df.itertuples(index=False), total=len(df)):
#     pkl_path = pkl_path_for(row)
#     if is_cached(pkl_path):
#         continue
#     img = cv2.cvtColor(
#         cv2.imread(f"{train_path}/{row.label}/{row.image}"),
#         cv2.COLOR_BGR2RGB,
#     )
#     if img is None:
#         misses.append((row.label, row.image))
#         continue
#     result = detect_landmarks(img)
#     atomic_write_pkl(pkl_path, result)
    
#     annotated = draw_landmarks_on_image(img, result)
#     preview.update(Image.fromarray(annotated))


In [ ]:
# def is_valid(idx):
#     row = df.iloc[idx]
#     label = row["label"]
#     image_name = row["image"]
#     result_path = f"train/pkl/{label}/{image_name}.pkl"
#     with open(result_path, "rb") as f:
#         result = pickle.load(f)
#     if label == "nothing":
#         return True
#     return bool(result.hand_landmarks)

# df["valid"] = [is_valid(idx) for idx in range(len(df))]
# valid_df = df[df["valid"] == True]
# valid_df.to_csv("./valid_train.csv", index=False)
df = pd.read_csv("./valid_train.csv")

In [ ]:
# Visualize the results

def load_result(idx):
    row = df.iloc[idx]
    label = row["label"]
    image_name = row["image"]
    result_path = f"train/pkl/{label}/{image_name}.pkl"
    
    with open(result_path, "rb") as f:
        result = pickle.load(f)            
        return result

unique_samples = df.drop_duplicates(subset=['label']).sort_values('label')

fig, axes = plt.subplots(5, 6, figsize=(12, 12))
flat_axes = axes.flatten()

for ax, (idx, row) in zip(flat_axes, unique_samples.iterrows()):
    label = row["label"]
    image_name = row["image"]
    image_path = f"{train_path}/{label}/{image_name}"
    result_path = f"train/pkl/{label}/{image_name}.pkl"
    
    with open(result_path, "rb") as f:
        result = pickle.load(f)        
        img = cv2.cvtColor(
            cv2.imread(image_path),
            cv2.COLOR_BGR2RGB,
        )        
        annotated = draw_landmarks_on_image(img, result)
        ax.imshow(annotated)
    ax.set_title(str(row["label"]))

for ax in flat_axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
df["label"].value_counts()

## Train the Fingerspelling Classifier

In [ ]:
import numpy as np
from sklearn.preprocessing import LabelEncoder

def extract_features(result):
    # Flatten 21 landmarks (x, y, z) = 63 features
    landmarks = result.hand_landmarks[0]
    return [coord for lm in landmarks for coord in (lm.x, lm.y, lm.z)]

X, y = [], []
for idx, row in df.iterrows():
    result = load_result(idx)
    if row["label"] == "nothing":
        X.append([0] * 63)
    else:
        X.append(extract_features(result))
    y.append(row["label"])

X = np.array(X)
le = LabelEncoder()
y_enc = le.fit_transform(y)

In [ ]:
import tensorflow as tf
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y_enc, test_size=0.2)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(63,)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(len(le.classes_), activation="softmax"),
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=30, batch_size=32)

In [ ]:
test_path = str(DATA_DIR / "asl_alphabet_test" / "asl_alphabet_test")

test_df = pd.DataFrame([
    {"image": f.split("/")[-1], "label": f.split("/")[-1].replace("_test.jpg", "")}
    for f in sorted(glob(f"{test_path}/*_test.jpg"))
])

for idx, row in test_df.iterrows():
    img = cv2.cvtColor(cv2.imread(f"{test_path}/{row.image}"), cv2.COLOR_BGR2RGB)
    result = detect_landmarks(img)
    if result.hand_landmarks:
        features = np.array(extract_features(result)).reshape(1, -1)
        pred = model.predict(features, verbose=0)
        test_df.at[idx, "predicted"] = le.inverse_transform([np.argmax(pred)])[0]
    else:
        test_df.at[idx, "predicted"] = "nothing"

fig, axes = plt.subplots(5, 6, figsize=(15, 12))
axes = axes.flatten()

for i, (idx, row) in enumerate(test_df.iterrows()):
    img = cv2.cvtColor(cv2.imread(f"{test_path}/{row.image}"), cv2.COLOR_BGR2RGB)
    result = detect_landmarks(img)
    annotated = draw_landmarks_on_image(img, result)
    correct = row["predicted"] == row["label"]
    axes[i].imshow(annotated)
    axes[i].set_title(f"True: {row['label']}\nPred: {row['predicted']}", color="green" if correct else "red")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
model.export("saved_model/")

## Export for the web

Convert the trained model to TensorFlow.js format so it can run in a browser.

In [ ]:
!pip install tensorflowjs

In [ ]:
!tensorflowjs_converter --input_format=tf_saved_model saved_model/ tfjs_model/